# SPD explorer (step 1)
Looks inside the three String Performance Dataset files copied to your Drive and writes a plain-text report back to your Drive (`SPD-explore-report.txt`). Run the cells in order. Nothing is uploaded anywhere else. Dataset terms: research and testing only until the authors confirm otherwise.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
root = '/content/drive/MyDrive'
files = sorted(glob.glob(root + '/SPD-*.zip'))
print(files)

In [ ]:
import zipfile, shutil
os.makedirs('/content/spd', exist_ok=True)
for f in files:
    name = os.path.basename(f)[:-4]
    out = '/content/spd/' + name
    if not os.path.isdir(out):
        shutil.copy(f, '/content/' + os.path.basename(f))
        with zipfile.ZipFile('/content/' + os.path.basename(f)) as z: z.extractall(out)
        os.remove('/content/' + os.path.basename(f))
    print('unpacked', name)

In [ ]:
import collections, json
rep = []
def say(*a):
    s = ' '.join(str(x) for x in a); print(s); rep.append(s)
for top in sorted(os.listdir('/content/spd')):
    base = '/content/spd/' + top
    say('=====', top)
    exts = collections.Counter(); size = 0; n = 0; dirs = collections.Counter()
    for dp, dn, fn in os.walk(base):
        depth = dp[len(base):].count('/')
        if depth <= 3: dirs[dp[len(base):] or '/'] = len(fn)
        for f in fn:
            exts[os.path.splitext(f)[1].lower()] += 1; size += os.path.getsize(os.path.join(dp, f)); n += 1
    say('files', n, 'size MB', round(size / 1e6), 'extensions', dict(exts))
    for d, c in list(dirs.items())[:25]: say('  dir', d, '->', c, 'files')
    ex = []
    for dp, dn, fn in os.walk(base):
        for f in sorted(fn)[:6]: ex.append(os.path.join(dp, f)[len(base):])
        if len(ex) > 30: break
    say('examples', ex[:30])

In [ ]:
import numpy as np, pickle
def peek(p):
    e = os.path.splitext(p)[1].lower()
    try:
        if e == '.json':
            d = json.load(open(p))
            say('JSON', p, 'type', type(d).__name__, 'keys' if isinstance(d, dict) else 'len', list(d)[:20] if isinstance(d, dict) else len(d))
        elif e == '.npy':
            a = np.load(p, allow_pickle=True); say('NPY', p, a.shape, a.dtype)
        elif e == '.npz':
            z = np.load(p, allow_pickle=True); say('NPZ', p, {k: z[k].shape for k in z.files})
        elif e in ('.pkl', '.pickle'):
            d = pickle.load(open(p, 'rb')); say('PKL', p, type(d).__name__, list(d)[:20] if isinstance(d, dict) else '')
        elif e in ('.csv', '.txt', '.md'):
            say('TEXT', p, open(p, errors='replace').read(600).replace(chr(10), ' | '))
    except Exception as ex:
        say('could not read', p, ex)
seen = collections.Counter()
for dp, dn, fn in os.walk('/content/spd'):
    for f in sorted(fn):
        e = os.path.splitext(f)[1].lower()
        if e in ('.json', '.npy', '.npz', '.pkl', '.pickle', '.csv', '.txt', '.md') and seen[e] < 3:
            seen[e] += 1; peek(os.path.join(dp, f))

In [ ]:
import subprocess
vids = []
for dp, dn, fn in os.walk('/content/spd'):
    for f in fn:
        if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv')): vids.append(os.path.join(dp, f))
say('videos found', len(vids))
for v in vids[:6]:
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'stream=width,height,r_frame_rate,nb_frames,duration', '-of', 'csv=p=0', v], capture_output=True, text=True)
    say('video', v[len('/content/spd'):], r.stdout.strip().replace(chr(10), ' ; '))
open(root + '/SPD-explore-report.txt', 'w').write('\n'.join(rep))
print('\nReport saved to your Drive: SPD-explore-report.txt')